# Web Scraping con Selenium



##### Extrae de las películas en cartelera datos. De ellas vamos a extraer la siguiente información:
- ##### Fecha de estreno
- ##### URL
- ##### Datos principales, como hemos hecho al principio
- ##### Nota media
- ##### Cantidad de votos
- ##### Críticas profesionales buenas, regulares y malas
- ##### Cinco primeras críticas

Importación de las librerías

In [7]:
pip install selenium webdriver-manager

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install undetected-chromedriver

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:

import pandas as pd


from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


from selenium.webdriver.common.by import By


from selenium.webdriver.common.keys import Keys


import time


from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


import undetected_chromedriver as uc


from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [10]:
service = Service(ChromeDriverManager().install())

In [11]:
from selenium import webdriver

driver = webdriver.Chrome()
print(driver.capabilities["browserVersion"])
driver.quit()

147.0.7727.138


Creamos el driver para controlar el navegador

In [18]:
driver = uc.Chrome(
    browser_executable_path=r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
    version_main=147
)


RECUERDA:

##### Beginner Selenium Cheatsheet:
Sacar un elemento:
- element = driver.find_element(by, value)

Sacar varios elementos:
- element = driver.find_elements(by, value)

Sacar atributos:
- attribute = element.--el atributo--
- attribute = element.get_attribute(--el atributo--)

Hacer click:
- element.click()

Teclear:
- element.send_keys()

Gestión de pestañas:
- driver.switch_to.window(driver.window_handles[-1])
- driver.get(url)
- driver.close()

Accedemos a la página principal

In [19]:

url = "https://www.filmaffinity.com"
driver.get(url)

Aceptamos el pop-up de ser necesario

In [20]:

accept = driver.find_element(By.XPATH, '//*[@id="accept-btn"]')



print("Etiqueta: {}".format(accept.tag_name))
print("Texto de la etiqueta: {}".format(accept.text))
print("Atributo mode: {}".format(accept.get_attribute('mode')))
print("Atributo size: {}".format(accept.get_attribute('size')))


accept.click()

Etiqueta: button
Texto de la etiqueta: ACEPTO
Atributo mode: primary
Atributo size: large


Hacemos una función que devuelva en un diccionario todos los datos de las películas, salvo la fecha de estreno y la url

Parámetros: url y fecha de estreno
Salida: Diccionario con todos los datos

In [21]:
def get_datos_pelicula(driver, url, estreno):

  
    movie_dict = dict()

    
    movie_dict['fecha_estreno'] = estreno
    movie_dict['url'] = url

    driver.get(url)
    time.sleep(2)

    
    principales = driver.find_element(by=By.CLASS_NAME, value='movie-info')

    dts = principales.find_elements(by=By.TAG_NAME, value='dt')
    dds = principales.find_elements(by=By.TAG_NAME, value='dd')

    i = 0
    while i < len(dts):
        movie_dict[dts[i].text] = dds[i].text
        i += 1

    
    nota_media = driver.find_elements(by=By.ID, value="movie-rat-avg")
    if len(nota_media) > 0:
        movie_dict['nota_media'] = nota_media[0].text

   
    cantidad_votos = driver.find_elements(by=By.CSS_SELECTOR, value="#movie-count-rat span")
    if len(cantidad_votos) > 0:
        movie_dict['cantidad_votos'] = cantidad_votos[0].text

  
    n_criticas = driver.find_elements(by=By.CSS_SELECTOR, value='#right-column > a > div > div.body > div > div.legend-wrapper .leg')

    if len(n_criticas) > 0:
        
        positivas = n_criticas[0].text
        movie_dict['positivas'] = positivas

        
        regulares = n_criticas[1].text
        movie_dict['regulares'] = regulares

        
        negativas = n_criticas[2].text
        movie_dict['negativas'] = negativas

    
    criticas = driver.find_elements(by=By.CSS_SELECTOR, value="ul#pro-reviews li")

    i = 0

    while i < 5 and i < len(criticas):
        critica = criticas[i].find_element(by=By.CSS_SELECTOR, value='div div').text
        movie_dict['critica_'+str(i)] = critica

        i += 1


    return movie_dict


Probamos la función que hemos hecho. Aquí tienes un enlace de prueba: https://www.filmaffinity.com/es/film599984.html

In [22]:
prueba = get_datos_pelicula(driver, "https://www.filmaffinity.com/es/film618375.html", "fecha")
prueba

{'fecha_estreno': 'fecha',
 'url': 'https://www.filmaffinity.com/es/film618375.html',
 'Título original': 'Oblivion',
 'Año': '2013',
 'Duración': '126 min.',
 'País': ' Estados Unidos',
 'Dirección': 'Joseph Kosinski',
 'Guion': 'Joseph Kosinski, Michael Arndt, Karl Gajdusek. Cómic: Joseph Kosinski, Arvid Nelson',
 'Reparto': 'Tom Cruise\nAndrea Riseborough\nOlga Kurylenko\nMorgan Freeman\nNikolaj Coster-Waldau\nZoe Bell',
 'Música': 'Anthony Gonzalez, M83, Joseph Trapanese',
 'Fotografía': 'Claudio Miranda',
 'Compañías': 'Universal Pictures, Chernin Entertainment, Relativity Studios, Monolith Pictures, Radical Studios',
 'Género': 'Ciencia ficción. Intriga | Futuro postapocalíptico. Distopía. Cómic',
 'Sinopsis': 'Año 2073. Hace más de 60 años la Tierra fue atacada; se ganó la guerra, pero la mitad del planeta quedó destruido, y todos los seres humanos fueron evacuados. Jack Harper (Tom Cruise), un antiguo marine, es uno de los últimos hombres que la habitan. Es un ingeniero de Dron

Entramos en el link de las películas en cartelera

In [24]:
driver.find_elements(By.PARTIAL_LINK_TEXT, value="Películas en cartelera")[0].click()

In [25]:
driver.find_elements(by=By.CSS_SELECTOR, value='.mt-content-wrapper div.row')

[]

Sacamos todas las películas y llamamos a la función con cada película

In [26]:
#Creamos una lista vacia
links = []

#Sacamos el elemento raíz
filas = driver.find_elements(by=By.CSS_SELECTOR, value='div.col a.poster-wrap')

for fila in filas:
    
    estreno = fila.find_element(by=By.CSS_SELECTOR, value='small.release-text').text
    
    link_pelicula = {
            'título': fila.get_attribute('title'),
            'url': fila.get_attribute('href'),
            'estreno': estreno.replace("\n", " ")
    }
    
    links.append(link_pelicula)



In [27]:
links

[{'título': 'Top Gun (Ídolos del aire) ',
  'url': 'https://www.filmaffinity.com/es/film383203.html',
  'estreno': '13 may.'},
 {'título': 'Top Gun: Maverick ',
  'url': 'https://www.filmaffinity.com/es/film395388.html',
  'estreno': '13 may.'},
 {'título': 'Las ovejas detectives ',
  'url': 'https://www.filmaffinity.com/es/film251675.html',
  'estreno': '8 may.'},
 {'título': 'Mortal Kombat II ',
  'url': 'https://www.filmaffinity.com/es/film120484.html',
  'estreno': '8 may.'},
 {'título': 'Couture (Alta costura) ',
  'url': 'https://www.filmaffinity.com/es/film726152.html',
  'estreno': '8 may.'},
 {'título': 'Yo no moriré de amor ',
  'url': 'https://www.filmaffinity.com/es/film493116.html',
  'estreno': '8 may.'},
 {'título': 'Bajo tus pies ',
  'url': 'https://www.filmaffinity.com/es/film182089.html',
  'estreno': '8 may.'},
 {'título': 'Recreación de un asesinato ',
  'url': 'https://www.filmaffinity.com/es/film445009.html',
  'estreno': '8 may.'},
 {'título': 'Billie Eilish. Hi

Ahora usamos los links para llamar a la funcion y sacar los datos

In [28]:

df = None

for link in links:
    datos_pelicula = get_datos_pelicula(driver, link['url'], link['estreno'])

   
    if df is None:
        df = pd.DataFrame(columns=datos_pelicula.keys())

    
    df = pd.concat([df, pd.DataFrame(datos_pelicula, index = [0])], ignore_index=True)
    print(f"Añadida {datos_pelicula['Título original']}")

Añadida Top Gunaka
Añadida Top Gun: Maverickaka
Añadida The Sheep Detectivesaka
Añadida Mortal Kombat II
Añadida Couturesaka
Añadida Yo no moriré de amor
Añadida Bajo tus pies
Añadida Re-creation
Añadida Billie Eilish. Hit Me Hard and Soft - The Tour (Live in 3D)
Añadida Los mejores años de nuestra vida
Añadida Deseo
Añadida Hangar rojo
Añadida La casa en el árbol
Añadida Escape inútil
Añadida Día ocho
Añadida The Devil Wears Prada 2
Añadida David
Añadida The Strangers: Chapter 3
Añadida Los justos
Añadida Gekijou-ban Tensei Shitara Slime Datta Ken: Soukai no Namida-hen
Añadida Amrum
Añadida Resurrectionaka
Añadida Allly baqi minkaka
Añadida The Plague
Añadida ¿Cómo hemos llegado a esto?
Añadida Enzo
Añadida Así chegou a noite
Añadida El renacer albirrojo
Añadida Michael
Añadida Kraken: El libro negro de las horas
Añadida La ahorcada
Añadida Cold Storage
Añadida In die Sonne schauenaka
Añadida Chien 51
Añadida Después de Kim
Añadida Casi todo bien
Añadida Marakudaaka
Añadida O Riso e a

In [29]:
df

,fecha_estreno,url,Título original,,Año,Duración,País,Dirección,Guion,Reparto,...,Género,Grupos,Sinopsis,nota_media,cantidad_votos,critica_0,critica_1,critica_2,critica_3,critica_4
0,13 may.,https://www.filmaffinity.com/es/film383203.html,Top Gunaka,,1986,110 min.,Estados Unidos,Tony Scott,"Jim Cash, Jack Epps Jr.",Tom Cruise\nKelly McGillis\nTom Skerritt\nAnth...,...,Acción. Drama. Romance | Drama romántico. Avio...,Top Gun,La Marina de los Estados Unidos ha creado una ...,"5,8",87.266,"El ""top"" Tom vuela y, de paso, enamora a la pr...","""Dirección de videoclip y mucha cara guapa, pa...","""Básicamente, es 'An Officer and a Gentleman' ...","""Las películas como esta son difíciles de crit...","""No es exactamente un clásico cinematográfico,..."
1,13 may.,https://www.filmaffinity.com/es/film395388.html,Top Gun: Maverickaka,,2022,131 min.,Estados Unidos,Joseph Kosinski,"Ehren Kruger, Eric Singer, Christopher McQuarr...",Tom Cruise\nMiles Teller\nJennifer Connelly\nJ...,...,Acción. Drama | Ejército. Aviones. Secuela,Top Gun,Después de más de treinta años de servicio com...,"7,0",30.565,"""Un delirio de unas dimensiones tan acertadas,...","""Más encanto, más aventura, más madurez y a má...","""Cruise pasa de la arrogancia a la épica (...)...","""Tom Cruise eleva el 'blockbuster' a la catego...","""Una película que juega abiertamente a la nost..."
2,8 may.,https://www.filmaffinity.com/es/film251675.html,The Sheep Detectivesaka,,2026,109 min.,Reino Unido,Kyle Balda,Craig Mazin. Novela: Leonie Swann,Hugh Jackman\nEmma Thompson\nNicholas Braun\nN...,...,Comedia. Intriga | Crimen. Animales. Whodunit,NaN,"Cada noche, un pastor lee en voz alta un asesi...","6,6",410,"""Es una encantadora producción a la que solame...","""Es cuando los borregos toman las riendas del ...","""Uno piensa que va a encontrarse con un dispar...","""Hay algo de la comedia visual y traviesa de '...","""Es una comedia animalista que habla de temas ..."
3,8 may.,https://www.filmaffinity.com/es/film120484.html,Mortal Kombat II,NaN,2026,116 min.,Estados Unidos,Simon McQuoid,"Jeremy Slater. Videojuego: Ed Boon, John Tobia...",Karl Urban\nAdeline Rudolph\nJessica McNamee\n...,...,Acción. Comedia | Artes marciales. Secuela. Vi...,Mortal Kombat,"Los campeones favoritos de los fans, ahora aco...","6,1",492,"""Lo que la salva -o nos salva- es el humor des...","""La mejor adaptación del videojuego jamás film...","""Esta secuela mejora la fórmula de la primera ...","""Su director por fin entiende que Mortal Komba...","""Un más de lo mismo en una secuela mediocre (...."
4,8 may.,https://www.filmaffinity.com/es/film726152.html,Couturesaka,,2025,106 min.,Francia,Alice Winocour,Alice Winocour,Angelina Jolie\nAnyier Anei\nElla Rumpf\nLouis...,...,Drama | Historias cruzadas. Moda. Enfermedad,NaN,En plena vorágine de la Semana de la Moda de P...,"5,6",179,"""Es una película irregular, brillante a veces,...","""Cruza historias en un vano intento de mostrar...","""Se ve en ella [Jolie] una entrega profunda (....","""En una película dirigida con elegancia, Alice...","""Carece de la mala leche de la fascinante 'Mod..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,27 mar.,https://www.filmaffinity.com/es/film878745.html,Zwei Staatsanwälteaka,,2025,118 min.,Alemania,Sergei Loznitsa,Sergei Loznitsa,Aleksandr Kuznetsov\nAnatoli Belyj\nVytautas K...,...,Drama | Años 30. Kafkiana. Histórico. Drama ju...,NaN,"Unión Soviética, 1937. Miles de cartas de dete...","6,6",318,"""Sergei Loznitsa desnuda las miserias del esta...","""Escalofriante y rotundo retrato de Serguei Lo...","""Ni el argumento ni su desarrollo sorprenden (...","""Lo que le otorga su extraordinaria capacidad ...","""Un mensaje escrito (...) catalizará una respu..."
62,20 mar.,https://www.filmaffinity.com/es/film310439.html,Amarga Navidad,NaN,2026,111 min.,España,Pedro Almodóvar,Pedro Almodóvar,Bárbara Lennie\nLeonardo Sbaraglia\nAitana Sán...,...,Drama | Amistad. Cine dentro del ci

In [30]:
df.columns

Index(['fecha_estreno', 'url', 'Título original', '', 'Año', 'Duración',
       'País', 'Dirección', 'Guion', 'Reparto', 'Música', 'Fotografía',
       'Compañías', 'Género', 'Grupos', 'Sinopsis', 'nota_media',
       'cantidad_votos', 'critica_0', 'critica_1', 'critica_2', 'critica_3',
       'critica_4'],
      dtype='str')